In [ ]:
!pip install pika


In [ ]:
import pika
import json
import os
import getpass

print("Pika imported successfully")

Pika imported successfully


In [ ]:
RABBITMQ_HOST = "129.153.75.221"
RABBITMQ_PORT = 5672
RABBITMQ_USERNAME = "bytesmart_interns"
RABBITMQ_VHOST = "/"

QUEUE_NAME = "student_wellness_predictions"

In [ ]:
RABBITMQ_PASSWORD = getpass.getpass(
    "Enter RabbitMQ password: "
)


Enter RabbitMQ password: ··········


In [ ]:
credentials = pika.PlainCredentials(
    RABBITMQ_USERNAME,
    RABBITMQ_PASSWORD
)

parameters = pika.ConnectionParameters(
    host=RABBITMQ_HOST,
    port=RABBITMQ_PORT,
    virtual_host=RABBITMQ_VHOST,
    credentials=credentials,
    heartbeat=60,
    blocked_connection_timeout=300
)

connection = pika.BlockingConnection(parameters)

channel = connection.channel()

print("Connected to Dev RabbitMQ successfully!")

Connected to Dev RabbitMQ successfully!


In [ ]:
QUEUE_NAME = "student_wellness_predictions"

channel.queue_declare(
    queue=QUEUE_NAME,
    durable=True
)

print(f"Queue '{QUEUE_NAME}' is ready.")

Queue 'student_wellness_predictions' is ready.


In [ ]:
def publish_message(message_data):

    message = json.dumps(message_data)

    channel.basic_publish(
        exchange="",
        routing_key=QUEUE_NAME,
        body=message,
        properties=pika.BasicProperties(
            delivery_mode=2,
            content_type="application/json"
        )
    )

    print("Message published successfully!")
    print(message)

In [ ]:
test_message = {
    "student_id": 10814,
    "predicted_risk": "Medium",
    "probabilities": {
        "Low": 1.18,
        "Medium": 62.01,
        "High": 36.81
    }
}

publish_message(test_message)

Message published successfully!
{"student_id": 10814, "predicted_risk": "Medium", "probabilities": {"Low": 1.18, "Medium": 62.01, "High": 36.81}}


In [ ]:
def process_message(message):

    student_id = message.get("student_id")
    predicted_risk = message.get("predicted_risk")
    probabilities = message.get("probabilities")

    print("\n========== MESSAGE PROCESSED ==========")
    print(f"Student ID     : {student_id}")
    print(f"Predicted Risk : {predicted_risk}")
    print(f"Probabilities  : {probabilities}")

    if predicted_risk == "High":
        print("Action         : Flag for early intervention review")

    elif predicted_risk == "Medium":
        print("Action         : Continue monitoring")

    elif predicted_risk == "Low":
        print("Action         : No immediate action required")

    print("========================================")

In [ ]:
def callback(ch, method, properties, body):

    try:

        print("\nMessage received from RabbitMQ")

        message = json.loads(body)

        process_message(message)

        ch.basic_ack(
            delivery_tag=method.delivery_tag
        )

    except Exception as e:

        print(f"Message processing failed: {e}")

        ch.basic_nack(
            delivery_tag=method.delivery_tag,
            requeue=False
        )

In [ ]:
channel.basic_qos(
    prefetch_count=1
)

channel.basic_consume(
    queue=QUEUE_NAME,
    on_message_callback=callback,
    auto_ack=False
)

print("Consumer is waiting for messages...")
print("Waiting for RabbitMQ messages...")

Consumer is waiting for messages...
Waiting for RabbitMQ messages...


In [ ]:
channel.start_consuming()


Message received from RabbitMQ

========== MESSAGE PROCESSED ==========
Student ID     : 10814
Predicted Risk : Medium
Probabilities  : {'Low': 1.18, 'Medium': 62.01, 'High': 36.81}
Action         : Continue monitoring

Message received from RabbitMQ

========== MESSAGE PROCESSED ==========
Student ID     : 1001
Predicted Risk : Medium
Probabilities  : {'Low': 0.23, 'Medium': 70.53, 'High': 29.25}
Action         : Continue monitoring

Message received from RabbitMQ

========== MESSAGE PROCESSED ==========
Student ID     : 1001
Predicted Risk : Medium
Probabilities  : {'Low': 0.23, 'Medium': 70.53, 'High': 29.25}
Action         : Continue monitoring
